In [28]:
import pandas as pd
import matplotlib.pyplot as plt

In [29]:
df = pd.read_csv("../data/raw/elspotprices_DK2_2024_01_01_2025_02_20.csv")

df["HourUTC"] = pd.to_datetime(df["HourUTC"])
df["HourDK"] = pd.to_datetime(df["HourDK"])

df = df.sort_values("HourUTC").reset_index(drop=True)

df.head()

,HourUTC,HourDK,PriceArea,SpotPriceDKK,SpotPriceEUR
0,2023-12-31 23:00:00,2024-01-01 00:00:00,DK2,217.160004,29.129999
1,2024-01-01 00:00:00,2024-01-01 01:00:00,DK2,212.160004,28.459999
2,2024-01-01 01:00:00,2024-01-01 02:00:00,DK2,198.740005,26.660000
3,2024-01-01 02:00:00,2024-01-01 03:00:00,DK2,182.490005,24.480000
4,2024-01-01 03:00:00,2024-01-01 04:00:00,DK2,178.990005,24.010000


In [30]:
df = df[["HourUTC","HourDK", "PriceArea", "SpotPriceEUR"]].copy()

df.head()

,HourUTC,HourDK,PriceArea,SpotPriceEUR
0,2023-12-31 23:00:00,2024-01-01 00:00:00,DK2,29.129999
1,2024-01-01 00:00:00,2024-01-01 01:00:00,DK2,28.459999
2,2024-01-01 01:00:00,2024-01-01 02:00:00,DK2,26.660000
3,2024-01-01 02:00:00,2024-01-01 03:00:00,DK2,24.480000
4,2024-01-01 03:00:00,2024-01-01 04:00:00,DK2,24.010000


In [31]:

df["hour"] = df["HourDK"].dt.hour
df["day_of_week"] = df["HourDK"].dt.dayofweek
df["month"] = df["HourDK"].dt.month
df["year"] = df["HourDK"].dt.year
df["is_weekend"] = df["HourDK"].dt.dayofweek.isin([5,6]).astype(int)

In [32]:
df["price_lag_24h"] = df["SpotPriceEUR"].shift(24)
df["price_lag_48h"] = df["SpotPriceEUR"].shift(48)
df["price_lag_168h"] = df["SpotPriceEUR"].shift(168)

In [33]:
df["price_rolling_mean_24h"] = df["SpotPriceEUR"].shift(1).rolling(window=24).mean()
df["price_rolling_std_24h"] = df["SpotPriceEUR"].shift(1).rolling(window=24).std()

df["price_rolling_mean_168h"] = df["SpotPriceEUR"].shift(1).rolling(window=168).mean()
df["price_rolling_std_168h"] = df["SpotPriceEUR"].shift(1).rolling(window=168).std()

In [34]:
df["target_next_hour"] = df["SpotPriceEUR"].shift(-1)

In [35]:
df_model = df.dropna().reset_index(drop=True)

df_model.head()

,HourUTC,HourDK,PriceArea,SpotPriceEUR,hour,day_of_week,month,year,is_weekend,price_lag_24h,price_lag_48h,price_lag_168h,price_rolling_mean_24h,price_rolling_std_24h,price_rolling_mean_168h,price_rolling_std_168h,target_next_hour
0,2024-01-07 23:00:00,2024-01-08 00:00:00,DK2,87.019997,0,0,1,2024,0,86.150002,84.879997,29.129999,88.660833,8.751415,87.877917,67.423618,84.320000
1,2024-01-08 00:00:00,2024-01-08 01:00:00,DK2,84.320000,1,0,1,2024,0,82.459999,83.199997,28.459999,88.697083,8.742360,88.222500,67.269329,81.820000
2,2024-01-08 01:00:00,2024-01-08 02:00:00,DK2,81.820000,2,0,1,2024,0,79.220001,82.029999,26.660000,88.774583,8.692768,88.555000,67.110030,79.110001
3,2024-01-08 02:00:00,2024-01-08 03:00:00,DK2,79.110001,3,0,1,2024,0,77.529999,82.180000,24.480000,88.882916,8.584039,88.883333,66.940116,78.480003
4,2024-01-08 03:00:00,2024-01-08 04:00:00,DK2,78.480003,4,0,1,2024,0,76.889999,82.529999,24.010000,88.948749,8.498820,89.208512,66.757829,82.099998


In [36]:
df_model.to_csv("../data/processed/dk2_price_features_next_hour.csv", index=False)

In [37]:
print("Rows:", len(df_model))
print("Columns:", df_model.columns.tolist())
print(df_model.isna().sum())

Rows: 9815
Columns: ['HourUTC', 'HourDK', 'PriceArea', 'SpotPriceEUR', 'hour', 'day_of_week', 'month', 'year', 'is_weekend', 'price_lag_24h', 'price_lag_48h', 'price_lag_168h', 'price_rolling_mean_24h', 'price_rolling_std_24h', 'price_rolling_mean_168h', 'price_rolling_std_168h', 'target_next_hour']
HourUTC                    0
HourDK                     0
PriceArea                  0
SpotPriceEUR               0
hour                       0
day_of_week                0
month                      0
year                       0
is_weekend                 0
price_lag_24h              0
price_lag_48h              0
price_lag_168h             0
price_rolling_mean_24h     0
price_rolling_std_24h      0
price_rolling_mean_168h    0
price_rolling_std_168h     0
target_next_hour           0
dtype: int64
